# Research Notebook for PISA question and answers on different languages

## Libraries

In [63]:
import os, getpass

In [64]:
from openai import OpenAI

In [65]:
# --- 0) Setup: imports & config
import os, re, time, json, math, random
from typing import Dict, List, Tuple, Any, Optional
import pandas as pd
import time

In [66]:
from tqdm import tqdm

## Configuration

In [67]:
BASE_URL = "https://ri-delta.ai/"  # <- replace with your portal URL root
MODEL_GPT = "gpt-5"                    # <- replace with exact model id if different
MODEL_CLAUDE = "claude-sonnet-4"
MODEL_GEMINI = "gemini-2.5-pro"

In [68]:
os.environ["LLM_API_KEY"] = "sk-8b736f91707c4d1cb2042dd74c0c647b"
client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=BASE_URL + "/api"    # e.g., https://llm.company.com/v1
)

In [69]:
SHEET_ID = "1QVPzB7uMwqJ6jCsHkwIILnXvDQIycpqkcV3bkiDpzyQ"
WORKSHEET_NAME = "dataset"  # change if needed
RESULTS_CSV = "llm_eval_results.csv"
SAMPLE_PER_LANGUAGE = 2     # 1 per language
MAX_LANGUAGES = 11          # 10 languages total
# LLM_TEMPERATURE = 0.2
# LLM_MAX_TOKENS = 256
SEED = 42

random.seed(SEED)

## Load data from Google Sheet

In [70]:
csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={WORKSHEET_NAME}"
try:
    df = pd.read_csv(csv_url)
except Exception as e:
    raise RuntimeError(
        "Failed to read the Google Sheet via CSV export. "
        "Make sure the sheet is shared as 'Anyone with the link can view', "
        f"ID is correct, and tab name matches. Underlying error: {e}"
    )

expected_cols = {
    "qid","language","question","context","options","gold",
    "answer_type","category","difficulty","rationale","source"
}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Your sheet is missing columns: {sorted(missing)}")

In [71]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   qid          105 non-null    object
 1   language     105 non-null    object
 2   question     105 non-null    object
 3   context      105 non-null    object
 4   options      105 non-null    object
 5   gold         105 non-null    object
 6   answer_type  105 non-null    object
 7   category     105 non-null    object
 8   difficulty   105 non-null    object
 9   rationale    44 non-null     object
 10  source       105 non-null    object
dtypes: object(11)
memory usage: 9.2+ KB


## Normalize & sample

In [72]:
df["language"] = df["language"].astype(str).str.strip()
lang_groups = []
for lang, sub in df.groupby("language", sort=True):
    lang_groups.append(sub.iloc[:SAMPLE_PER_LANGUAGE])

In [73]:
sampled = pd.concat(lang_groups, ignore_index=True).iloc[:MAX_LANGUAGES]
if sampled.empty:
    raise ValueError("No rows selected. Check your data.")

In [74]:
print(f"Selected {len(sampled)} rows across {sampled['language'].nunique()} languages.")
display(sampled[["qid","language","question","gold"]])

Selected 11 rows across 6 languages.


,qid,language,question,gold
0,q0006,Albanian,Çfarë përmendin shkencëtarët në artikull për t...,B
1,q0007,Albanian,Çfarë provash paraqesin Carl Lipo dhe Terry Hu...,D
2,q0001,Arabic,إذا قررتْ دانا شراء السيارة (د) وباعتها بعد ثل...,C
3,q0002,Arabic,في حالة استمرار المبيعات بهذا الشكل، في أي عام...,C
4,q0001,Chinese,如果譚雅決定購買汽車 D 並於三年後在保持良好狀態下轉售，那麼這輛汽車的大約轉售價格將是多少...,C
5,q0002,Chinese,如果這銷售趨勢持續下去的話，根據該模型，哪一年DVD 銷量會少於 1 百萬張？,C
6,q0001,Czech,Jaká bude přibližná prodejní cena auta (v zede...,C
7,q0002,Czech,"Pokud vývoj prodeje bude pokračovat, v kterém ...",C
8,q0001,English,If Tania decides to buy car D and resell it af...,C
9,q0002,English,"If this sales trend continues, what will be th...",C


## Parse options

In [75]:
def parse_options(raw: str) -> dict:
    """
    Parse options when stored as a JSON list of labeled strings, e.g.:
        ["A) 1575", "B) 8925", "C) 9000", "D) 9975"]

    Returns a dict like:
        {"A": "1575", "B": "8925", "C": "9000", "D": "9975"}
    """
    if pd.isna(raw):
        raise ValueError("Options are empty")

    try:
        # Try to load as JSON list
        items = json.loads(raw)
        if not isinstance(items, list):
            raise ValueError("Expected a list of options")
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON format for options: {e}")

    options = {}
    for item in items:
        if not isinstance(item, str):
            raise ValueError(f"Option is not a string: {item}")
        # Match patterns like "A) text", "B. text", or "C: text"
        match = re.match(r"^\s*([A-Z])[\)\.\:]\s*(.+)$", item.strip())
        if match:
            label, text = match.groups()
            options[label.upper()] = text.strip()
        else:
            # Fallback: assign next available letter automatically
            next_label = chr(ord('A') + len(options))
            options[next_label] = item.strip()

    return options


In [76]:
test = parse_options('["A) 2018", "B) 2019", "C) 2020", "D) 2021"]')
print(test)

{'A': '2018', 'B': '2019', 'C': '2020', 'D': '2021'}


In [77]:
options_block = "\n".join([f"{k}. {v}" for k,v in test.items()])
print(options_block)

A. 2018
B. 2019
C. 2020
D. 2021


## Build prompt

In [78]:
def build_prompt(row: pd.Series, options: Dict[str,str]) -> str:
    """
    Builds a language-agnostic but context-aware prompt for MCQ.
    """
    options_block = "\n".join([f"{k}. {v}" for k,v in options.items()])
    # You can adapt language instruction if you want the rationale in the same language:
    lang = str(row["language"]).strip()

    return (
        # f"You are answering a multiple-choice question. "
        # f"Return ONLY the chosen option letter (A, B, C, ...). "
        f"{row['context']}\n"
        f"{row['question']}\n"
        f"{options_block}\n"
        # f"\nReply format STRICTLY:\n" 
        # f"<LETTER>\n" 
    )

In [79]:
answer_letter_regex = re.compile(r"<\s*([A-Z])\s*[.)]?\s*>")

def extract_letter(text: str, valid_letters: List[str]) -> str:
    """
    Extract the first single-letter A-Z token that is in valid_letters.
    """
    if not text:
        return ""
    # First line is the letter per our format; but still be defensive:
    first_line = text.splitlines()[0].strip().rstrip(".)").upper()
    # If first line is a single valid letter, use it
    if len(first_line) == 1 and first_line.upper() in valid_letters:
        return first_line.upper()
    # Else find any A-Z token
    m = answer_letter_regex.search(text.upper())
    if m and m.group(1) in valid_letters:
        return m.group(1)
    return ""

## LLM call wrapper

In [80]:
def llm_completion(
    prompt: str,
    model: Optional[str] = None,
    # temperature: float = 0.2,
    # max_tokens: int = 256,
    system_prompt: str = "You are a helpful assistant that answers multiple-choice questions. Reply format: <LETTER>.",
    retries: int = 2,
    backoff_seconds: float = 1.5,
    **kwargs,
) -> str:
    """
    Call an LLM via chat.completions and return text.
    - `prompt`: user content (string)
    - `model`: overrides global MODEL_NAME if provided
    - `temperature`, `max_tokens`: usual decoding controls
    - `system_prompt`: system role content
    - `retries`: retry on transient errors
    - `backoff_seconds`: base backoff between retries
    - `**kwargs`: forwarded to client.chat.completions.create (e.g., stop, seed)
    """
    mdl = model or MODEL_CLAUDE
    last_err = None

    for attempt in range(retries + 1):
        try:
            # Build arguments dynamically — include only if provided
            call_args = dict(
                model=mdl,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
                stream=False
            )

            # Add optional params only if explicitly set
            if "temperature" in kwargs and kwargs["temperature"] is not None:
                call_args["temperature"] = kwargs["temperature"]
            if "max_tokens" in kwargs and kwargs["max_tokens"] is not None:
                call_args["max_tokens"] = kwargs["max_tokens"]

            # Merge other kwargs (e.g., stop, seed, etc.)
            for k, v in kwargs.items():
                if k not in call_args:
                    call_args[k] = v

            # Actual model call
            response = client.chat.completions.create(**call_args)
            if response is None:
                return "No response from model."
            elif not hasattr(response, "choices") or not response.choices:
                return "Response object missing 'choices'."
            else:
                return(response.choices[0].message.content or "").strip()

        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(backoff_seconds * (attempt + 1))
            else:
                raise

## Evaluation loop

In [81]:
def eval_rows(rows: pd.DataFrame, model_name: str = None, sleep_s: float = 0.0, retries: int = 2) -> pd.DataFrame:
    results = []
    for i, row in tqdm(rows.iterrows(), total=len(rows), desc=f"Evaluating {model_name}"):
        qid = row["qid"]
        lang = row["language"]
        gold = str(row["gold"]).strip().upper()

        # Parse options
        try:
            opts = parse_options(row["options"])
        except Exception as e:
            results.append({
                "qid": qid, "language": lang, "pred": "",
                "gold": gold, "is_correct": False,
                "error": f"OptionsParseError: {e}", "raw": ""
            })
            continue

        valid_letters = sorted(list(opts.keys()))
        prompt = build_prompt(row, opts)

        # call model with simple retry
        raw = ""
        err = ""
        for attempt in range(retries + 1):
            try:
                raw = llm_completion(prompt, model = model_name)
                break
            except Exception as e:
                err = f"{type(e).__name__}: {e}"
                if attempt < retries:
                    time.sleep(1.5 * (attempt + 1))
                else:
                    raw = ""
        pred = extract_letter(raw, valid_letters)
        is_correct = (pred == gold)

        results.append({
            "qid": qid,
            "language": lang,
            "pred": pred,
            "gold": gold,
            "is_correct": bool(is_correct),
            "error": err,
            "raw": raw,
            "question": row["question"],
            "options_json": json.dumps(opts, ensure_ascii=False),
        })
        if sleep_s > 0:
            time.sleep(sleep_s)
    return pd.DataFrame(results)

## Execution and results

In [82]:
MODELS_TO_TEST = [
    ("GPT", MODEL_GPT),        # (tag, model_name_for_API)
    ("Claude", MODEL_CLAUDE),
    ("Gemini", MODEL_GEMINI)
]

In [83]:
filtered_df = df[df["language"] == "Georgian"]

In [84]:
N_CYCLES = 2  #
all_results = []

for tag, model_name in MODELS_TO_TEST:
    print(f"\nEvaluating {tag} -> {model_name}")
    for cycle in range(1, N_CYCLES + 1):
        print(f"  • cycle {cycle}/{N_CYCLES}")
        t0 = time.time()

        df_model = eval_rows(df, model_name=model_name) # sampled or df or filtered_df
        df_model["model_tag"] = tag
        df_model["model_name"] = model_name
        df_model["cycle"]      = cycle
        all_results.append(df_model)


Evaluating GPT -> gpt-5
  • cycle 1/2


Evaluating gpt-5: 100%|██████████| 105/105 [07:43<00:00,  4.41s/it]


  • cycle 2/2


Evaluating gpt-5: 100%|██████████| 105/105 [08:12<00:00,  4.69s/it]



Evaluating Claude -> claude-sonnet-4
  • cycle 1/2


...

Evaluating claude-sonnet-4: 100%|██████████| 105/105 [20:08<00:00, 11.51s/it]



Evaluating Gemini -> gemini-2.5-pro
  • cycle 1/2


Evaluating gemini-2.5-pro: 100%|██████████| 105/105 [16:32<00:00,  9.45s/it]


  • cycle 2/2


Evaluating gemini-2.5-pro: 100%|██████████| 105/105 [16:49<00:00,  9.62s/it]


In [85]:
res_df = pd.concat(all_results, ignore_index=True)

In [86]:
overall_by_model = (
    res_df.groupby(["model_tag","model_name"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by model:")
display(overall_by_model)


Overall accuracy by model:


,model_tag,model_name,accuracy
1,GPT,gpt-5,0.976190
2,Gemini,gemini-2.5-pro,0.852381
0,Claude,claude-sonnet-4,0.838095


In [87]:
overall_by_lang = (
    res_df.groupby(["language"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by lang:")
display(overall_by_lang)


Overall accuracy by lang:


,language,accuracy
2,Chinese,0.950000
4,English,0.933333
3,Czech,0.916667
10,Turkish,0.916667
8,Russian,0.916667
6,Kazakh,0.916667
7,Mongolian,0.866667
9,Thai,0.850000
0,Albanian,0.850000
1,Arabic,0.833333


In [88]:
by_model_lang = (
    res_df.groupby(["model_tag","model_name","language"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values(["model_tag","language"])
)
print("\nAccuracy by model & language:")
display(by_model_lang)


Accuracy by model & language:


,model_tag,model_name,language,accuracy
0,Claude,claude-sonnet-4,Albanian,0.85
1,Claude,claude-sonnet-4,Arabic,0.80
2,Claude,claude-sonnet-4,Chinese,1.00
3,Claude,claude-sonnet-4,Czech,0.85
4,Claude,claude-sonnet-4,English,0.90
5,Claude,claude-sonnet-4,Georgian,0.75
6,Claude,claude-sonnet-4,Kazakh,0.85
7,Claude,claude-sonnet-4,Mongolian,0.80
8,Claude,claude-sonnet-4,Russian,0.85
9,Claude,claude-sonnet-4,Thai,0.75


In [93]:
by_question = (
    res_df.groupby(["model_tag","model_name","language","question"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values(["accuracy"])
)
print("\nAccuracy by question:")
display(by_question)


Accuracy by question:


,model_tag,model_name,language,question,accuracy
280,Gemini,gemini-2.5-pro,Mongolian,"Хэрэв Туяа Г машиныг худалдан аваад, хамгийн с...",0.0
284,Gemini,gemini-2.5-pro,Russian,Если Таня решит купить автомобиль D и перепрод...,0.0
259,Gemini,gemini-2.5-pro,Georgian,ეს დებულება ფაქტია თუ მოსაზრება?\nბოლოდროინდელ...,0.0
261,Gemini,gemini-2.5-pro,Georgian,"თუ ქეთი გადაწყვეტს, იყიდოს მანქანა D და გაყიდო...",0.0
269,Gemini,gemini-2.5-pro,Kazakh,Егер Тоғжан Г автокөлігін сатып алуды шешіп жә...,0.0
...,...,...,...,...,...
308,Gemini,gemini-2.5-pro,Turkish,Metnin temel amacı nedir?,1.0
309,Gemini,gemini-2.5-pro,Turkish,Neptün gezegenin Güneş'e ortalama uzaklığı yak...,1.0
1,Claude,claude-sonnet-4,Albanian,Cili është numri më i madh i kutive mesatare q...,1.0
311,Gemini,gemini-2.5-pro,Turkish,USÜD'e (IDFA) göre önde gelen sağlık uzmanları...,1.0


In [89]:
def safe_acc(s):
    return float('nan') if s.empty else s.mean()
overall_acc = safe_acc(res_df["is_correct"])
print(f"\nCombined overall accuracy on {len(res_df)} items: {overall_acc:.3f}")


Combined overall accuracy on 630 items: 0.889


In [90]:
# One combined file + one per-model file
RESULTS_COMBINED_CSV = "llm_eval_results__combined.csv"
res_df.to_csv(RESULTS_COMBINED_CSV, index=False)
print(f"Saved combined results to: {RESULTS_COMBINED_CSV}")

for tag, _ in MODELS_TO_TEST:
    out_path = f"llm_eval_results__{tag}.csv"
    res_df.query("model_tag == @tag").to_csv(out_path, index=False)
    print(f"Saved {tag} results to: {out_path}")

# (Optional) quick peek
display(res_df.head())

Saved combined results to: llm_eval_results__combined.csv
Saved GPT results to: llm_eval_results__GPT.csv
Saved Claude results to: llm_eval_results__Claude.csv
Saved Gemini results to: llm_eval_results__Gemini.csv


,qid,language,pred,gold,is_correct,error,raw,question,options_json,model_tag,model_name,cycle
0,q0001,English,C,C,True,,C,If Tania decides to buy car D and resell it af...,"{""A"": ""1575"", ""B"": ""8925"", ""C"": ""9000"", ""D"": ""...",GPT,gpt-5,1
1,q0002,English,C,C,True,,C.,"If this sales trend continues, what will be th...","{""A"": ""2018"", ""B"": ""2019"", ""C"": ""2020"", ""D"": ""...",GPT,gpt-5,1
2,q0003,English,B,B,True,,B,What is the greatest number of medium boxes th...,"{""A"": ""320"", ""B"": ""128"", ""C"": ""96"", ""D"": ""64""}",GPT,gpt-5,1
3,q0004,English,D,D,True,,D,"On average, approximately how many million kil...","{""A"": ""180 million km"", ""B"": ""450 million km"",...",GPT,gpt-5,1
4,q0005,English,A,A,True,,A.,Given the average margin of victory for the se...,"{""A"": ""Yes"", ""B"": ""No""}",GPT,gpt-5,1
